# Week 38: Slater determinants as bit patterns, full configuration interaction, and the start of Hartree-Fock theory

FYS4480/FYS9480, September 17-18, 2026. Companion notebook to `WeeklySlides/week38.pdf`.

Everything below is self-contained numpy. The code mirrors `BookPrograms/chapter03/wick.py`, `BookPrograms/chapter05/fci.py` and `BookPrograms/chapter06/hartreefock.py`, but is written so that every identity stated on the slides is checked numerically here:

1. Slater determinants as bit patterns, and $\hat H|\Phi\rangle$ by bit operations (section 3.16)
2. The pairing model as the first FCI model: 70 determinants against the $6\times6$ seniority-zero matrix
3. The eigenvalue problem from the variational principle: the Rayleigh quotient, its gradient, the Ritz bound, and the characteristic polynomial
4. The structure of the Hamiltonian matrix, read off from the Condon-Slater rules
5. Truncations and size consistency; six levels and six particles (924 determinants)
6. The projected FCI equations: the energy equation, the singles and doubles equations, term by term against the exact eigenvector (the equations behind the diagrams)
7. The "non-practical" iterative solution of the amplitude equations
8. Hartree-Fock: the energy functional of one determinant, the self-consistent field loop, Brillouin's theorem, and FCI in the Hartree-Fock basis

In [1]:
import numpy as np
from itertools import combinations
from math import comb
np.set_printoptions(precision=6, suppress=True, linewidth=120)
rng = np.random.default_rng(2038)

## 1. Slater determinants as bit patterns (section 3.16)

A determinant with $N$ particles in $n$ orbitals is an integer with $N$ bits set, $\Phi_{3,6,10,13}=a_3^\dagger a_6^\dagger a_{10}^\dagger a_{13}^\dagger|0\rangle = |0001001000100100\rangle$. A creation operator $a_p^\dagger$ returns zero if bit $p$ is set, otherwise sets it and returns the phase $(-1)^{l}$ with $l$ the number of set bits *before* $p$ (the operators it has to pass). The same $l$ for $a_p$. The pair move $a_j^\dagger a_i$ has phase $(-1)^{\#\text{occupied strictly between }i\text{ and }j}$.

In [2]:
def popcount(x):
    return bin(x).count("1")

def bits(state, n):
    '''Book notation: alpha_0 leftmost.'''
    return "|" + "".join(str((state >> p) & 1) for p in range(n)) + ">"

def annihilate_bit(state, p):
    if not (state >> p) & 1:
        return 0, None
    return (-1) ** popcount(state & ((1 << p) - 1)), state & ~(1 << p)

def create_bit(state, p):
    if (state >> p) & 1:
        return 0, None
    return (-1) ** popcount(state & ((1 << p) - 1)), state | (1 << p)

def apply_string(state, ops):
    '''ops = [(index, dagger), ...] leftmost first; returns (phase, state) or (0, None).'''
    phase = 1
    for index, dagger in reversed(ops):
        s, state = (create_bit if dagger else annihilate_bit)(state, index)
        if s == 0:
            return 0, None
        phase *= s
    return phase, state

c = lambda p: (p, True)      # a_p^dagger
d = lambda p: (p, False)     # a_p

def move(state, i, j):
    '''a_j^dagger a_i on a bit pattern, phase by a mask and a popcount (the slide).'''
    if not (state >> i) & 1:  return 0, None
    if i == j:                return 1, state
    if (state >> j) & 1:      return 0, None
    lo, hi = min(i, j), max(i, j)
    between = state & ((1 << hi) - (1 << (lo + 1)))
    return (-1 if popcount(between) % 2 else 1), (state & ~(1 << i)) | (1 << j)

state = (1 << 3) | (1 << 6) | (1 << 10) | (1 << 13)
print("Phi_{3,6,10,13} =", bits(state, 16), "=", state)
print("a_4^+ Phi       =", create_bit(state, 4)[0], bits(create_bit(state, 4)[1], 16))
print("a_6^+ Phi       =", create_bit(state, 6))
print("a_4^+ a_0 Phi_{0,3,6,10,13}: phase", move(state | 1, 0, 4)[0], bits(move(state | 1, 0, 4)[1], 16))
print("dim for n = 16, N = 4:", comb(16, 4))

# the table of the slide: a_2^+ on five-bit patterns
for pat in ["00111", "01011", "01101", "10011", "10101", "10110", "11001", "11010"]:
    st = sum(1 << k for k, ch in enumerate(pat) if ch == "1")
    s, new = create_bit(st, 2)
    print(f"a_2^+ |{pat}> = ", "0" if s == 0 else f"({s:+d}) {bits(new, 5)}")

# the move phase equals the product of the two single-operator phases, for every pattern of 5 bits
ok = True
for st in range(32):
    for i in range(5):
        for j in range(5):
            s1, n1 = apply_string(st, [c(j), d(i)])
            s2, n2 = move(st, i, j)
            ok &= (s1 == s2) and (n1 == n2)
print("move() == a_j^+ a_i by two bit operations, all 32 x 25 cases:", ok)

Phi_{3,6,10,13} = |0001001000100100> = 9288
a_4^+ Phi       = -1 |0001101000100100>
a_6^+ Phi       = (0, None)
a_4^+ a_0 Phi_{0,3,6,10,13}: phase -1 |0001101000100100>
dim for n = 16, N = 4: 1820
a_2^+ |00111> =  0
a_2^+ |01011> =  (-1) |01111>
a_2^+ |01101> =  0
a_2^+ |10011> =  (-1) |10111>
a_2^+ |10101> =  0
a_2^+ |10110> =  0
a_2^+ |11001> =  (+1) |11101>
a_2^+ |11010> =  (+1) |11110>
move() == a_j^+ a_i by two bit operations, all 32 x 25 cases: True


$\hat H|\Phi\rangle$ by bit operations: loop over the *occupied* orbitals for the annihilators and over all orbitals for the creators. Checked against the explicit Fock-space matrices of week 37 for every determinant of a small space.

In [3]:
def apply_hamiltonian(state, h, vAS):
    '''H|state> as a dictionary {new_state: amplitude}, by bit operations only.'''
    n = h.shape[0]
    out = {}
    occ = [p for p in range(n) if (state >> p) & 1]
    for q in occ:                                    # one-body: a_p^+ a_q, q occupied
        for p in range(n):
            if h[p, q] != 0.0:
                s, new = apply_string(state, [c(p), d(q)])
                if s: out[new] = out.get(new, 0.0) + s * h[p, q]
    for r in occ:                                    # two-body: 1/4 <pq|v|rs> a_p^+ a_q^+ a_s a_r
        for s_ in occ:
            if s_ == r: continue
            for p in range(n):
                for q in range(n):
                    if vAS[p, q, r, s_] != 0.0:
                        s, new = apply_string(state, [c(p), c(q), d(s_), d(r)])
                        if s: out[new] = out.get(new, 0.0) + 0.25 * s * vAS[p, q, r, s_]
    return out

# Fock-space matrices (week 37) for the check
class FockSpace:
    def __init__(self, n):
        self.n, self.dim = n, 1 << n
        self._a = [self._annihilate(p) for p in range(n)]
        self._adag = [M.T.copy() for M in self._a]
    def _annihilate(self, p):
        M = np.zeros((self.dim, self.dim))
        for st in range(self.dim):
            if st & (1 << p):
                M[st ^ (1 << p), st] = (-1) ** popcount(st & ((1 << p) - 1))
        return M
    def create(self, p): return self._adag[p]
    def annihilate(self, p): return self._a[p]

def random_onebody(n):
    h = rng.normal(size=(n, n)); return (h + h.T) / 2

def random_twobody(n):
    v = rng.normal(size=(n, n, n, n))
    v = v + v.transpose(1, 0, 3, 2); v = v + v.transpose(2, 3, 0, 1)
    return v - v.transpose(0, 1, 3, 2)          # antisymmetrised

def fock_hamiltonian(space, h, vAS):
    n = space.n
    H = sum(h[p, q] * space.create(p) @ space.annihilate(q) for p in range(n) for q in range(n))
    for p in range(n):
        for q in range(n):
            cc = space.create(p) @ space.create(q)
            for r in range(n):
                for s in range(n):
                    if vAS[p, q, r, s] != 0.0:
                        H = H + 0.25 * vAS[p, q, r, s] * cc @ space.annihilate(s) @ space.annihilate(r)
    return H

n = 6
space = FockSpace(n)
h0, vAS = random_onebody(n), random_twobody(n)
Hfock = fock_hamiltonian(space, h0, vAS)
ok = True
for occ in combinations(range(n), 3):
    st = sum(1 << p for p in occ)
    vec = np.zeros(space.dim)
    for new, amp in apply_hamiltonian(st, h0, vAS).items():
        vec[new] += amp
    unit = np.zeros(space.dim); unit[st] = 1.0
    ok &= np.allclose(vec, Hfock @ unit)
print("H|Phi> by bit operations == 64 x 64 matrix-vector product, all C(6,3) = 20 determinants:", ok)

H|Phi> by bit operations == 64 x 64 matrix-vector product, all C(6,3) = 20 determinants: True


## 2. The pairing model as the first FCI model (sections 5.1-5.6)

`SlaterBasis` enumerates all $\binom{n}{N}$ determinants as bit patterns, sorted by excitation level relative to a reference, and builds the Hamiltonian matrix (or a truncation of it) with `apply_hamiltonian`. Four doubly degenerate levels, four particles, $\xi=1$: the $70\times70$ matrix against the $6\times6$ seniority-zero matrix of chapter 4.

In [4]:
class SlaterBasis:
    '''All determinants with N particles in n orbitals, as bit patterns, ordered by excitation level.'''
    def __init__(self, n, N, reference):
        self.n, self.N, self.reference = n, N, reference
        states = [sum(1 << p for p in occ) for occ in combinations(range(n), N)]
        self.level = {s: popcount(s & ~reference) for s in states}
        self.states = sorted(states, key=lambda s: (self.level[s], s))
        self.index = {s: k for k, s in enumerate(self.states)}
    def levels(self):
        return [self.level[s] for s in self.states]
    def hamiltonian(self, h, vAS, levels=None):
        '''Matrix in the full space, or restricted to the excitation levels in `levels` (e.g. {0, 2} for CID).'''
        keep = [s for s in self.states if levels is None or self.level[s] in levels]
        idx = {s: k for k, s in enumerate(keep)}
        Hm = np.zeros((len(keep), len(keep)))
        for s in keep:
            for new, amp in apply_hamiltonian(s, h, vAS).items():
                if new in idx:
                    Hm[idx[new], idx[s]] += amp
        return Hm, keep

def pairing_elements(levels, xi=1.0, g=1.0):
    '''One-body h and antisymmetrised two-body elements of the pairing Hamiltonian; orbital 2p = (p,+), 2p+1 = (p,-).'''
    m = 2 * levels
    h = np.zeros((m, m)); v = np.zeros((m, m, m, m))
    for p in range(levels):
        for s in (0, 1):
            h[2*p+s, 2*p+s] = xi * p
    for p in range(levels):
        for q in range(levels):
            pu, pd, qu, qd = 2*p, 2*p+1, 2*q, 2*q+1
            v[pu, pd, qu, qd] = v[pd, pu, qd, qu] = -g / 2
            v[pu, pd, qd, qu] = v[pd, pu, qu, qd] = +g / 2
    return h, v

def seniority_zero_matrix(g, levels=4, xi=1.0):
    pairs = list(combinations(range(1, levels + 1), 2))
    M = np.zeros((len(pairs), len(pairs)))
    for A, (p, q) in enumerate(pairs):
        M[A, A] = 2 * xi * ((p - 1) + (q - 1)) - g
        for B, (r, s) in enumerate(pairs):
            if A != B and len({p, q} & {r, s}) == 1:
                M[A, B] = -g / 2
    return M

levels, xi = 4, 1.0
basis4 = SlaterBasis(2 * levels, 4, reference=0b1111)
lev4 = np.array(basis4.levels())
print("dimension C(8,4) =", len(basis4.states), "; per excitation level:", [int((lev4 == k).sum()) for k in range(5)])
hp, vp = pairing_elements(levels, xi, 1.0)
Hm, _ = basis4.hamiltonian(hp, vp)
print("non-zero blocks (rows/columns = excitation level 0..4):")
for k in range(5):
    print("  ", k, ["x" if np.abs(Hm[np.ix_(lev4 == k, lev4 == l)]).max() > 1e-12 else "0" for l in range(5)])
print("number of non-zero matrix elements out of 70 x 70 = 4900:", int((np.abs(Hm) > 1e-12).sum()))

print(f"\n{'g':>6} {'E_ref':>10} {'E0 (6x6)':>12} {'E0 (70x70)':>12} {'|broken-pair amplitude|':>24}")
for g in [-1.0, -0.5, 0.0, 0.5, 1.0]:
    hp, vp = pairing_elements(levels, xi, g)
    Hm, keep = basis4.hamiltonian(hp, vp)
    E, C = np.linalg.eigh(Hm)
    E6 = np.linalg.eigvalsh(seniority_zero_matrix(g))[0]
    broken = [k for k, s in enumerate(keep) if any(((s >> (2*p)) & 1) != ((s >> (2*p+1)) & 1) for p in range(levels))]
    print(f"{g:>6.2f} {Hm[0,0]:>10.6f} {E6:>12.8f} {E[0]:>12.8f} {np.abs(C[broken, 0]).max():>24.2e}")

dimension C(8,4) = 70 ; per excitation level: [1, 16, 36, 16, 1]
non-zero blocks (rows/columns = excitation level 0..4):
   0 ['x', '0', 'x', '0', '0']
   1 ['0', 'x', '0', 'x', '0']
   2 ['x', '0', 'x', '0', 'x']
   3 ['0', 'x', '0', 'x', '0']
   4 ['0', '0', 'x', '0', 'x']
number of non-zero matrix elements out of 70 x 70 = 4900: 142

     g      E_ref     E0 (6x6)   E0 (70x70)  |broken-pair amplitude|
 -1.00   3.000000   2.77987014   2.77987014                 1.53e-16
 -0.50   2.500000   2.43688426   2.43688426                 6.25e-17
  0.00   2.000000   2.00000000   2.00000000                 0.00e+00
  0.50   1.500000   1.41677428   1.41677428                 5.55e-17
  1.00   1.000000   0.63554847   0.63554847                 1.25e-16


## 3. The eigenvalue problem from the variational principle (section 5.3)

With $|\Psi\rangle=\sum_K C_K|\Phi_K\rangle$ in an orthonormal determinant basis the energy is the **Rayleigh quotient** $E[\mathbf C]=\mathbf C^\dagger\mathbf H\mathbf C/\mathbf C^\dagger\mathbf C$. Its gradient with respect to $C_K^*$ is $2\,[\mathbf H\mathbf C-E\mathbf C]_K/\mathbf C^\dagger\mathbf C$, so a stationary point is an eigenvector, and the Lagrange multiplier of the constrained problem is the eigenvalue. Checked here numerically: the gradient vanishes at the eigenvectors and nowhere else, $E[\mathbf C]\ge E_0$ for random $\mathbf C$ (the Ritz bound), and the eigenvalues are the roots of the characteristic polynomial $\det(\mathbf H-E\mathbf 1)=0$.

In [5]:
g = 1.0
H6 = seniority_zero_matrix(g)
E6, C6 = np.linalg.eigh(H6)

def rayleigh(H, C):
    return (C @ H @ C) / (C @ C)

def gradient(H, C):
    '''dE/dC_K^* for real C: 2 (H C - E C)_K / (C.C).'''
    return 2 * (H @ C - rayleigh(H, C) * C) / (C @ C)

print("eigenvalues of the 6x6 matrix at g = 1:", E6)
print("|gradient| at each eigenvector          :", [f"{np.linalg.norm(gradient(H6, C6[:, k])):.1e}" for k in range(6)])
trial = rng.normal(size=(2000, 6))
Rq = np.array([rayleigh(H6, t) for t in trial])
print(f"Ritz bound: min over 2000 random trial vectors = {Rq.min():.6f} >= E_0 = {E6[0]:.6f}; max = {Rq.max():.6f} <= E_5 = {E6[-1]:.6f}")
print("smallest |gradient| among the random vectors :", f"{min(np.linalg.norm(gradient(H6, t)) for t in trial):.2e}  (never zero away from an eigenvector)")

# steepest descent on the Rayleigh quotient converges to the lowest eigenpair: optimisation view of the Ritz problem
Cvec = rng.normal(size=6)
for it in range(500):
    Cvec = Cvec - 0.1 * gradient(H6, Cvec)
    Cvec /= np.linalg.norm(Cvec)
print(f"steepest descent on E[C], 500 steps of size 0.1: E = {rayleigh(H6, Cvec):.8f}, exact E_0 = {E6[0]:.8f}, overlap with eigenvector = {abs(Cvec @ C6[:, 0]):.10f}")

# the characteristic polynomial det(H - E 1) = 0
coeffs = np.poly(H6)                     # coefficients of det(E 1 - H), highest power first
roots = np.sort(np.roots(coeffs).real)
print("\ncharacteristic polynomial, coefficients of E^6 ... E^0:", coeffs)
print("its roots       :", roots)
print("eigh eigenvalues:", E6)
print("max |root - eigenvalue| =", f"{np.abs(roots - E6).max():.1e}")

# the 2x2 case in closed form: one pair in two levels
def two_level(g, xi=1.0):
    return np.array([[-g/2, -g/2], [-g/2, 2*xi - g/2]])
for g2 in (0.5, 1.0):
    Eexact = xi - g2/2 - np.sqrt(xi**2 + g2**2/4)
    print(f"g = {g2}: 2x2 lowest root xi - g/2 - sqrt(xi^2 + g^2/4) = {Eexact:.8f}, eigh = {np.linalg.eigvalsh(two_level(g2))[0]:.8f}")

eigenvalues of the 6x6 matrix at g = 1: [0.635548 2.935381 5.       5.       7.20894  9.22013 ]
|gradient| at each eigenvector          : ['5.3e-16', '3.4e-15', '5.6e-15', '1.8e-15', '3.4e-15', '3.9e-15']
Ritz bound: min over 2000 random trial vectors = 1.026184 >= E_0 = 0.635548; max = 8.942987 <= E_5 = 9.220130
smallest |gradient| among the random vectors : 3.58e-01  (never zero away from an eigenvector)
steepest descent on E[C], 500 steps of size 0.1: E = 0.63554847, exact E_0 = 0.63554847, overlap with eigenvector = 1.0000000000

characteristic polynomial, coefficients of E^6 ... E^0: [    1.   -30.   352. -2038.  5979. -7940.  3100.]
its roots       : [0.635548 2.935381 5.       5.       7.20894  9.22013 ]
eigh eigenvalues: [0.635548 2.935381 5.       5.       7.20894  9.22013 ]
max |root - eigenvalue| = 3.4e-13
g = 0.5: 2x2 lowest root xi - g/2 - sqrt(xi^2 + g^2/4) = -0.28077641, eigh = -0.28077641
g = 1.0: 2x2 lowest root xi - g/2 - sqrt(xi^2 + g^2/4) = -0.61803399, eigh = -0.61

Why nobody solves the characteristic polynomial for anything but a $2\times2$ or $3\times3$ example: the roots of a polynomial are exquisitely sensitive to its coefficients. Below, the coefficients of the degree-70 polynomial of the pairing matrix are perturbed at the level of double-precision rounding, and the roots move by orders of magnitude more than the eigenvalues do under the same perturbation of the matrix.

In [6]:
hp, vp = pairing_elements(4, 1.0, 1.0)
H70, _ = basis4.hamiltonian(hp, vp)
E70 = np.linalg.eigvalsh(H70)
coeffs70 = np.poly(H70)
roots70 = np.sort(np.roots(coeffs70).real)
print(f"degree-70 characteristic polynomial: largest coefficient {np.abs(coeffs70).max():.2e}")
print(f"max |root - eigenvalue| from the exact coefficients          : {np.abs(roots70 - E70).max():.2e}   (already useless: the coefficients span 55 orders of magnitude)")
pert = coeffs70 * (1 + 1e-14 * rng.normal(size=coeffs70.size))
roots_p = np.sort(np.roots(pert).real)
print(f"max |root - eigenvalue| after a 1e-14 relative coefficient noise: {np.abs(roots_p - E70).max():.2e}")
Hp_ = H70 + 1e-14 * np.abs(H70).max() * rng.normal(size=H70.shape); Hp_ = (Hp_ + Hp_.T) / 2
print(f"max eigenvalue shift for a 1e-14 relative perturbation of the matrix : {np.abs(np.linalg.eigvalsh(Hp_) - E70).max():.2e}")

degree-70 characteristic polynomial: largest coefficient 9.61e+54
max |root - eigenvalue| from the exact coefficients          : 1.33e+01   (already useless: the coefficients span 55 orders of magnitude)
max |root - eigenvalue| after a 1e-14 relative coefficient noise: 1.44e+01
max eigenvalue shift for a 1e-14 relative perturbation of the matrix : 4.30e-13


## 4. The structure of the Hamiltonian matrix from the Condon-Slater rules (section 5.4)

For a random two-body Hamiltonian (no symmetry to hide anything) with $n=8$, $N=4$ — one-body energies $0,1,\dots,7$ plus a random mixing, and a random antisymmetrised interaction scaled so that the reference is a sensible starting point: the matrix is banded in the excitation level with bandwidth two, its first row is $E_{\rm ref}$, $f_{ai}$, $\langle ab|v|ij\rangle$, and every block beyond the band is zero.

In [7]:
n8, N4 = 8, 4
h8 = np.diag(np.arange(n8, dtype=float)) + 0.2 * random_onebody(n8)     # a 'shell structure' plus a random mixing
v8 = 0.15 * random_twobody(n8)                                          # a moderate two-body interaction
ref8 = 0b00001111
basis8 = SlaterBasis(n8, N4, ref8)
lev8 = np.array(basis8.levels())
H8, keep8 = basis8.hamiltonian(h8, v8)
print("dimension", len(keep8), "; per level", [int((lev8 == k).sum()) for k in range(5)])
print("non-zero blocks:")
for k in range(5):
    print("  ", k, ["x" if np.abs(H8[np.ix_(lev8 == k, lev8 == l)]).max() > 1e-12 else "0" for l in range(5)])

holes = [0, 1, 2, 3]; parts = [4, 5, 6, 7]
E_ref = sum(h8[i, i] for i in holes) + 0.5 * sum(v8[i, j, i, j] for i in holes for j in holes)
f8 = h8 + sum(v8[:, i, :, i] for i in holes)
print(f"\n<Phi_0|H|Phi_0> = {H8[0, 0]:.6f},  E_ref = sum h_ii + 1/2 sum <ij|v|ij> = {E_ref:.6f}")
def det_of(parts_, holes_):
    st = ref8
    for i in holes_: st &= ~(1 << i)
    for a in parts_: st |= (1 << a)
    return st
def sign_of(parts_, holes_):
    '''phase of a_a^+ a_b^+ ... a_j a_i |Phi_0> relative to the bit pattern.'''
    s, _ = apply_string(ref8, [c(a) for a in parts_] + [d(i) for i in reversed(holes_)])
    return s
ok1 = all(np.isclose(sign_of([a], [i]) * H8[basis8.index[det_of([a], [i])], 0], f8[a, i]) for a in parts for i in holes)
ok2 = all(np.isclose(sign_of([a, b], [i, j]) * H8[basis8.index[det_of([a, b], [i, j])], 0], v8[a, b, i, j])
          for a in parts for b in parts if a < b for i in holes for j in holes if i < j)
print("<Phi_i^a|H|Phi_0> = f_ai for all 16 singles            :", ok1)
print("<Phi_ij^ab|H|Phi_0> = <ab|v|ij> for all 36 doubles     :", ok2)
print("<Phi_ijk^abc|H|Phi_0> = 0 for all 16 triples          :", np.abs(H8[lev8 == 3, 0]).max() < 1e-12)

dimension 70 ; per level [1, 16, 36, 16, 1]
non-zero blocks:
   0 ['x', 'x', 'x', '0', '0']
   1 ['x', 'x', 'x', 'x', '0']
   2 ['x', 'x', 'x', 'x', 'x']
   3 ['0', 'x', 'x', 'x', 'x']
   4 ['0', '0', 'x', 'x', 'x']

<Phi_0|H|Phi_0> = 6.497174,  E_ref = sum h_ii + 1/2 sum <ij|v|ij> = 6.497174
<Phi_i^a|H|Phi_0> = f_ai for all 16 singles            : True
<Phi_ij^ab|H|Phi_0> = <ab|v|ij> for all 36 doubles     : True
<Phi_ijk^abc|H|Phi_0> = 0 for all 16 triples          : True


## 5. Truncations and size consistency (sections 5.8-5.9)

CIS, CID, CISD against FCI for the four-level pairing model, then six levels and six particles ($\binom{12}{6}=924$ determinants, Figure 5.2 of the lecture notes), and the size-consistency test on two non-interacting copies of a two-level system.

In [8]:
schemes = {"CIS": {0, 1}, "CID": {0, 2}, "CISD": {0, 1, 2}, "FCI": None}
print("four levels, four particles")
print(f"{'g':>6} {'E_ref':>10} " + " ".join(f"{k:>12}" for k in schemes) + "   dims")
for g in [0.25, 0.5, 1.0, 2.0]:
    hp, vp = pairing_elements(4, 1.0, g)
    energies, dims = [], []
    for name, lv in schemes.items():
        Hm, keep = basis4.hamiltonian(hp, vp, lv)
        energies.append(np.linalg.eigvalsh(Hm)[0]); dims.append(len(keep))
    E_ref_g = basis4.hamiltonian(hp, vp, {0})[0][0, 0]
    print(f"{g:>6.2f} {E_ref_g:>10.6f} " + " ".join(f"{E:>12.8f}" for E in energies) + f"   {dims}")

four levels, four particles
     g      E_ref          CIS          CID         CISD          FCI   dims
  0.25   1.750000   1.75000000   1.73050700   1.73050700   1.73045985   [17, 37, 53, 70]


  0.50   1.500000   1.50000000   1.41759645   1.41759645   1.41677428   [17, 37, 53, 70]
  1.00   1.000000   1.00000000   0.64890655   0.64890655   0.63554847   [17, 37, 53, 70]


  2.00   0.000000   0.00000000  -1.35208670  -1.35208670  -1.48965216   [17, 37, 53, 70]


In [9]:
basis6 = SlaterBasis(12, 6, reference=0b111111)
lev6 = np.array(basis6.levels())
print("six levels, six particles: dimension", len(basis6.states), "; per level", [int((lev6 == k).sum()) for k in range(7)])
schemes6 = {"CIS": {0, 1}, "CISD": {0, 1, 2}, "CISDTQ": {0, 1, 2, 3, 4}, "FCI": None}
print(f"{'g':>6} {'E_ref':>10} " + " ".join(f"{k:>14}" for k in schemes6) + "   CISD %corr   CISDTQ %corr")
for g in [0.05, 0.25, 0.5, 1.0, 2.0]:
    hp, vp = pairing_elements(6, 1.0, g)
    res = {}
    for name, lv in schemes6.items():
        Hm, keep = basis6.hamiltonian(hp, vp, lv)
        res[name] = (np.linalg.eigvalsh(Hm)[0], len(keep))
    E_ref_g = basis6.hamiltonian(hp, vp, {0})[0][0, 0]
    corr = res["FCI"][0] - E_ref_g
    print(f"{g:>6.2f} {E_ref_g:>10.6f} " + " ".join(f"{res[k][0]:>14.8f}" for k in schemes6)
          + f"   {100*(res['CISD'][0]-E_ref_g)/corr:>9.2f}   {100*(res['CISDTQ'][0]-E_ref_g)/corr:>11.4f}")
print("dimensions:", {k: res[k][1] for k in schemes6})

six levels, six particles: dimension 924 ; per level [1, 36, 225, 400, 225, 36, 1]
     g      E_ref            CIS           CISD         CISDTQ            FCI   CISD %corr   CISDTQ %corr


  0.05   5.925000     5.92500000     5.92381898     5.92381873     5.92381873       99.98      100.0000


  0.25   5.625000     5.62500000     5.59295380     5.59276099     5.59276086       99.40       99.9996


  0.50   5.250000     5.25000000     5.10969074     5.10593698     5.10592638       97.39       99.9926


  1.00   4.500000     4.50000000     3.87227845     3.80238641     3.80152797       89.87       99.8771


  2.00   3.000000     3.00000000     0.54353367    -0.15967891    -0.18915925       77.03       99.0756
dimensions: {'CIS': 37, 'CISD': 262, 'CISDTQ': 887, 'FCI': 924}


In [10]:
def copies_elements(ncopies, xi=1.0, g=1.0):
    '''ncopies independent two-level pairing systems; orbitals 4c..4c+3 belong to copy c.'''
    m = 4 * ncopies
    h = np.zeros((m, m)); v = np.zeros((m, m, m, m))
    for cpy in range(ncopies):
        o = 4 * cpy
        for s in (0, 1):
            h[o + 2 + s, o + 2 + s] = xi
        for p in range(2):
            for q in range(2):
                pu, pd, qu, qd = o+2*p, o+2*p+1, o+2*q, o+2*q+1
                v[pu, pd, qu, qd] = v[pd, pu, qd, qu] = -g / 2
                v[pu, pd, qd, qu] = v[pd, pu, qu, qd] = +g / 2
    return h, v

print(f"{'g':>6} {'2 E(A)':>14} {'E(AB) FCI':>14} {'E(AB) CID':>14} {'CID error':>12} {'E(AB) CISDTQ':>14}")
for g in [0.25, 0.5, 1.0]:
    h1, v1 = copies_elements(1, 1.0, g)
    EA = np.linalg.eigvalsh(SlaterBasis(4, 2, 0b0011).hamiltonian(h1, v1)[0])[0]
    h2, v2 = copies_elements(2, 1.0, g)
    b2 = SlaterBasis(8, 4, 0b00110011)
    EAB = np.linalg.eigvalsh(b2.hamiltonian(h2, v2)[0])[0]
    ECID = np.linalg.eigvalsh(b2.hamiltonian(h2, v2, {0, 2})[0])[0]
    EQ = np.linalg.eigvalsh(b2.hamiltonian(h2, v2, {0, 1, 2, 3, 4})[0])[0]
    print(f"{g:>6.2f} {2*EA:>14.8f} {EAB:>14.8f} {ECID:>14.8f} {ECID-EAB:>12.2e} {EQ:>14.8f}")
print("closed form for one copy: E(A) = xi - g/2 - sqrt(xi^2 + g^2/4) =", 1 - 0.5 - np.sqrt(1 + 0.25), "at g = 1")

# the exponential ansatz on two copies: (1 + t A_2^A)(1 + t A_2^B)|Phi_0> is exact with ONE parameter
h2, v2 = copies_elements(2, 1.0, 1.0)
b2 = SlaterBasis(8, 4, 0b00110011)
H2, keep2 = b2.hamiltonian(h2, v2)
EAB = np.linalg.eigvalsh(H2)[0]
idx = {s: k for k, s in enumerate(keep2)}
ref, A_only, B_only, both = 0b00110011, 0b00111100, 0b11000011, 0b11001100
def cc_energy(t):
    vec = np.zeros(len(keep2)); vec[idx[ref]] = 1; vec[idx[A_only]] = t; vec[idx[B_only]] = t; vec[idx[both]] = t*t
    return (vec @ H2 @ vec) / (vec @ vec)
ts = np.linspace(-2, 2, 40001); Ets = [cc_energy(t) for t in ts]
print(f"exp(T2) ansatz, one parameter t: min_t E = {min(Ets):.8f} at t = {ts[int(np.argmin(Ets))]:.4f};  FCI E(AB) = {EAB:.8f}")

     g         2 E(A)      E(AB) FCI      E(AB) CID    CID error   E(AB) CISDTQ
  0.25    -0.26556444    -0.26556444    -0.26550480     5.96e-05    -0.26556444
  0.50    -0.56155281    -0.56155281    -0.56066017     8.93e-04    -0.56155281
  1.00    -1.23606798    -1.23606798    -1.22474487     1.13e-02    -1.23606798
closed form for one copy: E(A) = xi - g/2 - sqrt(xi^2 + g^2/4) = -0.6180339887498949 at g = 1


exp(T2) ansatz, one parameter t: min_t E = -1.23606797 at t = 0.2361;  FCI E(AB) = -1.23606798


## 6. The projected FCI equations, term by term (the equations behind the diagrams)

With $\hat H=E_{\rm ref}+\hat F_N+\hat V_N$, intermediate normalisation $|\Psi\rangle=(1+\hat C_1+\hat C_2+\hat C_3+\hat C_4+\cdots)|\Phi_0\rangle$ and antisymmetric amplitudes ($\hat C_2=\tfrac14\sum C_{ij}^{ab}a_a^\dagger a_b^\dagger a_j a_i$, $\hat C_3=\tfrac1{36}\sum\cdots$), projecting $(\hat H-E)|\Psi\rangle=0$ onto $\langle\Phi_0|$, $\langle\Phi_i^a|$ and $\langle\Phi_{ij}^{ab}|$ gives, with $\Delta E=E-E_{\rm ref}$,

$$\Delta E=\sum_{ia}f_{ia}C_i^a+\tfrac14\sum_{ijab}\langle ij|v|ab\rangle C_{ij}^{ab},$$

$$\Delta E\,C_i^a = f_{ai}+\sum_b f_{ab}C_i^b-\sum_j f_{ji}C_j^a+\sum_{jb}\langle aj|v|ib\rangle C_j^b
+\sum_{jb}f_{jb}C_{ij}^{ab}+\tfrac12\sum_{jbc}\langle aj|v|bc\rangle C_{ij}^{bc}-\tfrac12\sum_{jkb}\langle jk|v|ib\rangle C_{jk}^{ab}
+\tfrac14\sum_{jkbc}\langle jk|v|bc\rangle C_{ijk}^{abc},$$

$$\Delta E\,C_{ij}^{ab}=\langle ab|v|ij\rangle+P(ab)\sum_c f_{bc}C_{ij}^{ac}-P(ij)\sum_k f_{kj}C_{ik}^{ab}+\tfrac12\sum_{cd}\langle ab|v|cd\rangle C_{ij}^{cd}+\tfrac12\sum_{kl}\langle kl|v|ij\rangle C_{kl}^{ab}+P(ij)P(ab)\sum_{kc}\langle kb|v|cj\rangle C_{ik}^{ac}
+[\text{singles}]+[\text{triples}]+[\text{quadruples}],$$

with $P(ab)X_{ab}=X_{ab}-X_{ba}$. Every term is one diagram on the slides. Here each term is evaluated with the exact FCI eigenvector of the random $n=8$, $N=4$ Hamiltonian of section 4 (which has singles, doubles, triples and quadruples) and the equations are verified to machine precision, including the singles, triples and quadruples pieces of the doubles equation:
$$[\text{singles}]=P(ij)\sum_c\langle ab|v|cj\rangle C_i^c-P(ab)\sum_k\langle kb|v|ij\rangle C_k^a+P(ij)P(ab)\,f_{ai}C_j^b,$$
$$[\text{triples}]=\sum_{kc}f_{kc}C_{ijk}^{abc}+\tfrac12P(ab)\sum_{kcd}\langle bk|v|cd\rangle C_{ijk}^{acd}-\tfrac12P(ij)\sum_{klc}\langle kl|v|jc\rangle C_{ikl}^{abc},\qquad
[\text{quadruples}]=\tfrac14\sum_{klcd}\langle kl|v|cd\rangle C_{ijkl}^{abcd}.$$

The last singles term, $P(ij)P(ab)f_{ai}C_j^b$, is a *disconnected* product of two lower pieces — the reference-to-singles element times a single amplitude; it is the kind of term that the exponential ansatz of coupled-cluster theory removes from the equations, and it is absent in a Hartree-Fock basis where $f_{ai}=0$.

In [11]:
E8, C8 = np.linalg.eigh(H8)
E0, vec = E8[0], C8[:, 0]
vec = vec / vec[0]                                   # intermediate normalisation, C_0 = 1
dE = E0 - E_ref

def amp(parts_, holes_):
    '''C_{ij..}^{ab..} with the sign of a_a^+ a_b^+ ... a_j a_i |Phi_0> (holes_ = (i, j, ...)).'''
    st = det_of(parts_, holes_)
    return sign_of(list(parts_), list(holes_)) * vec[basis8.index[st]]

nP, nH = len(parts), len(holes)
C1 = np.zeros((n8, n8)); C2 = np.zeros((n8,)*4); C3 = np.zeros((n8,)*6); C4 = np.zeros((n8,)*8)
from itertools import permutations
def perm_sign(p):
    s = 1
    p = list(p)
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            if p[i] > p[j]: s = -s
    return s
for a in parts:
    for i in holes:
        C1[a, i] = amp([a], [i])
for a, b in combinations(parts, 2):
    for i, j in combinations(holes, 2):
        x = amp([a, b], [i, j])
        C2[a, b, i, j] = C2[b, a, j, i] = x; C2[b, a, i, j] = C2[a, b, j, i] = -x
for pa in combinations(parts, 3):
    for ho in combinations(holes, 3):
        x = amp(list(pa), list(ho))
        for pp in permutations(range(3)):
            for ph in permutations(range(3)):
                C3[pa[pp[0]], pa[pp[1]], pa[pp[2]], ho[ph[0]], ho[ph[1]], ho[ph[2]]] = perm_sign(pp) * perm_sign(ph) * x
for pa in combinations(parts, 4):
    for ho in combinations(holes, 4):
        x = amp(list(pa), list(ho))
        for pp in permutations(range(4)):
            for ph in permutations(range(4)):
                C4[tuple(pa[k] for k in pp) + tuple(ho[k] for k in ph)] = perm_sign(pp) * perm_sign(ph) * x

f = f8; v = v8
P, Hh = parts, holes
# energy equation
dE_eq = sum(f[i, a] * C1[a, i] for i in Hh for a in P) + 0.25 * sum(v[i, j, a, b] * C2[a, b, i, j] for i in Hh for j in Hh for a in P for b in P)
print(f"energy equation: Delta E = {dE:.10f}, sum f_ia C_i^a + 1/4 sum <ij|v|ab> C_ij^ab = {dE_eq:.10f}")
print(f"   singles part {sum(f[i, a] * C1[a, i] for i in Hh for a in P):.6f}, doubles part {dE_eq - sum(f[i, a] * C1[a, i] for i in Hh for a in P):.6f}")

# singles equation, term by term
def singles_terms(a, i):
    return {
        "f_ai":                       f[a, i],
        "sum_b f_ab C_i^b":           sum(f[a, b] * C1[b, i] for b in P),
        "-sum_j f_ji C_j^a":         -sum(f[j, i] * C1[a, j] for j in Hh),
        "sum_jb <aj|v|ib> C_j^b":     sum(v[a, j, i, b] * C1[b, j] for j in Hh for b in P),
        "sum_jb f_jb C_ij^ab":        sum(f[j, b] * C2[a, b, i, j] for j in Hh for b in P),
        "1/2 sum <aj|v|bc> C_ij^bc":  0.5 * sum(v[a, j, b, cc] * C2[b, cc, i, j] for j in Hh for b in P for cc in P),
        "-1/2 sum <jk|v|ib> C_jk^ab": -0.5 * sum(v[j, k, i, b] * C2[a, b, j, k] for j in Hh for k in Hh for b in P),
        "1/4 sum <jk|v|bc> C_ijk^abc": 0.25 * sum(v[j, k, b, cc] * C3[a, b, cc, i, j, k] for j in Hh for k in Hh for b in P for cc in P),
    }
worst = 0.0
for a in P:
    for i in Hh:
        t = singles_terms(a, i)
        worst = max(worst, abs(sum(t.values()) - dE * C1[a, i]))
print(f"singles equations, all 16 (a,i): max |lhs - Delta E C_i^a| = {worst:.1e}")
t = singles_terms(P[0], Hh[0])
print(f"  the eight terms for a = {P[0]}, i = {Hh[0]}:")
for k, x in t.items():
    print(f"    {k:<30} {x:>12.6f}")
print(f"    {'sum':<30} {sum(t.values()):>12.6f}   Delta E C_i^a = {dE * C1[P[0], Hh[0]]:.6f}")

energy equation: Delta E = -5.7494604184, sum f_ia C_i^a + 1/4 sum <ij|v|ab> C_ij^ab = -5.7494604184
   singles part -4.847624, doubles part -0.901837
singles equations, all 16 (a,i): max |lhs - Delta E C_i^a| = 8.0e-15
  the eight terms for a = 4, i = 0:
    f_ai                              -0.158381
    sum_b f_ab C_i^b                  -0.358418
    -sum_j f_ji C_j^a                  1.111001
    sum_jb <aj|v|ib> C_j^b            -0.704327
    sum_jb f_jb C_ij^ab                0.730845
    1/2 sum <aj|v|bc> C_ij^bc          0.210481
    -1/2 sum <jk|v|ib> C_jk^ab         0.238577
    1/4 sum <jk|v|bc> C_ijk^abc        0.132900
    sum                                1.202677   Delta E C_i^a = 1.202677


In [12]:
def doubles_terms(a, b, i, j):
    Pab = lambda X: X(a, b) - X(b, a)
    Pij = lambda X: X(i, j) - X(j, i)
    T = {}
    T["<ab|v|ij>"] = v[a, b, i, j]
    T["P(ab) sum_c f_bc C_ij^ac"] = Pab(lambda a_, b_: sum(f[b_, cc] * C2[a_, cc, i, j] for cc in P))
    T["-P(ij) sum_k f_kj C_ik^ab"] = -Pij(lambda i_, j_: sum(f[k, j_] * C2[a, b, i_, k] for k in Hh))
    T["1/2 sum <ab|v|cd> C_ij^cd"] = 0.5 * sum(v[a, b, cc, dd] * C2[cc, dd, i, j] for cc in P for dd in P)
    T["1/2 sum <kl|v|ij> C_kl^ab"] = 0.5 * sum(v[k, l, i, j] * C2[a, b, k, l] for k in Hh for l in Hh)
    T["P(ij)P(ab) sum <kb|v|cj> C_ik^ac"] = sum(sg1 * sg2 * sum(v[k, b_, cc, j_] * C2[a_, cc, i_, k] for k in Hh for cc in P)
                                               for sg1, (i_, j_) in ((1, (i, j)), (-1, (j, i))) for sg2, (a_, b_) in ((1, (a, b)), (-1, (b, a))))
    T["P(ij) sum_c <ab|v|cj> C_i^c"] = Pij(lambda i_, j_: sum(v[a, b, cc, j_] * C1[cc, i_] for cc in P))
    T["-P(ab) sum_k <kb|v|ij> C_k^a"] = -Pab(lambda a_, b_: sum(v[k, b_, i, j] * C1[a_, k] for k in Hh))
    T["P(ij)P(ab) f_ai C_j^b  (disconnected)"] = f[a, i] * C1[b, j] - f[a, j] * C1[b, i] - f[b, i] * C1[a, j] + f[b, j] * C1[a, i]
    T["sum_kc f_kc C_ijk^abc"] = sum(f[k, cc] * C3[a, b, cc, i, j, k] for k in Hh for cc in P)
    T["1/2 P(ab) sum <bk|v|cd> C_ijk^acd"] = 0.5 * Pab(lambda a_, b_: sum(v[b_, k, cc, dd] * C3[a_, cc, dd, i, j, k] for k in Hh for cc in P for dd in P))
    T["-1/2 P(ij) sum <kl|v|jc> C_ikl^abc"] = -0.5 * Pij(lambda i_, j_: sum(v[k, l, j_, cc] * C3[a, b, cc, i_, k, l] for k in Hh for l in Hh for cc in P))
    T["1/4 sum <kl|v|cd> C_ijkl^abcd"] = 0.25 * sum(v[k, l, cc, dd] * C4[a, b, cc, dd, i, j, k, l] for k in Hh for l in Hh for cc in P for dd in P)
    return T

worst = 0.0
for a, b in combinations(P, 2):
    for i, j in combinations(Hh, 2):
        T = doubles_terms(a, b, i, j)
        worst = max(worst, abs(sum(T.values()) - dE * C2[a, b, i, j]))
print(f"doubles equations, all 36 (ab,ij): max |lhs - Delta E C_ij^ab| = {worst:.1e}")
T = doubles_terms(P[0], P[1], Hh[0], Hh[1])
print(f"  the thirteen terms for ab = {P[0]}{P[1]}, ij = {Hh[0]}{Hh[1]}:")
for k, x in T.items():
    print(f"    {k:<38} {x:>12.6f}")
print(f"    {'sum':<38} {sum(T.values()):>12.6f}   Delta E C_ij^ab = {dE * C2[P[0], P[1], Hh[0], Hh[1]]:.6f}")

doubles equations, all 36 (ab,ij): max |lhs - Delta E C_ij^ab| = 9.1e-15
  the thirteen terms for ab = 45, ij = 01:
    <ab|v|ij>                                 -0.137071
    P(ab) sum_c f_bc C_ij^ac                  -0.377233
    -P(ij) sum_k f_kj C_ik^ab                  0.415133
    1/2 sum <ab|v|cd> C_ij^cd                  0.105914
    1/2 sum <kl|v|ij> C_kl^ab                 -0.022570
    P(ij)P(ab) sum <kb|v|cj> C_ik^ac           0.178085
    P(ij) sum_c <ab|v|cj> C_i^c                0.219289
    -P(ab) sum_k <kb|v|ij> C_k^a               0.236753
    P(ij)P(ab) f_ai C_j^b  (disconnected)     -0.568303
    sum_kc f_kc C_ijk^abc                      0.150703
    1/2 P(ab) sum <bk|v|cd> C_ijk^acd          0.038796
    -1/2 P(ij) sum <kl|v|jc> C_ikl^abc         0.007151
    1/4 sum <kl|v|cd> C_ijkl^abcd              0.004873
    sum                                        0.251518   Delta E C_ij^ab = 0.251518


## 7. The non-practical way: solving the amplitude equations by iteration (section 5.10)

Rewrite the singles and doubles equations as $C_H^P = \bigl[\langle\Phi_H^P|\hat H|\Phi_0\rangle+\sum_{P'H'\ne PH}\langle\Phi_H^P|\hat H|\Phi_{H'}^{P'}\rangle C_{H'}^{P'}\bigr]/\bigl(E-\langle\Phi_H^P|\hat H|\Phi_H^P\rangle\bigr)$ and iterate with $E$ from the energy equation, starting from the first-order perturbative guess $C_i^a=f_{ai}/(\varepsilon_i-\varepsilon_a)$, $C_{ij}^{ab}=\langle ab|v|ij\rangle/(\varepsilon_i+\varepsilon_j-\varepsilon_a-\varepsilon_b)$ — the "non-practical" route of the slides. For the pairing model with four levels the iteration (all excitation levels kept, so it must converge to FCI) does converge, and each iterate is a partial summation that the later chapters identify with an order of perturbation theory.

In [13]:
def iterate_amplitudes(Hm, nit=60, damping=1.0):
    '''Jacobi-type iteration of the projected equations in the full space (index 0 = reference).'''
    dim = Hm.shape[0]
    diag = np.diag(Hm)
    Cv = np.zeros(dim); Cv[0] = 1.0
    E = Hm[0, 0]
    # first-order guess: C_K = H_K0 / (E_ref - H_KK)
    for K in range(1, dim):
        Cv[K] = Hm[K, 0] / (Hm[0, 0] - diag[K]) if abs(Hm[0, 0] - diag[K]) > 1e-12 else 0.0
    history = []
    for it in range(nit):
        E = Hm[0, 0] + Hm[0, 1:] @ Cv[1:]                       # energy equation
        new = Cv.copy()
        for K in range(1, dim):
            rhs = Hm[K, 0] + Hm[K, 1:] @ Cv[1:] - diag[K] * Cv[K]
            new[K] = rhs / (E - diag[K])
        Cv = damping * new + (1 - damping) * Cv
        history.append(E)
    return E, Cv, history

for g in [0.25, 0.5, 1.0]:
    hp, vp = pairing_elements(4, 1.0, g)
    Hm, keep = basis4.hamiltonian(hp, vp)
    Efci = np.linalg.eigvalsh(Hm)[0]
    E, Cv, hist = iterate_amplitudes(Hm, nit=80)
    E2 = Hm[0, 0] - 7/24 * g**2
    print(f"g = {g}: iterate 1 (second-order PT) {hist[0]:.8f} [E_ref - 7/24 g^2 = {E2:.8f}], iterate 5 {hist[4]:.8f}, iterate 20 {hist[19]:.8f}, iterate 80 {hist[-1]:.8f}, FCI {Efci:.8f}")

g = 0.25: iterate 1 (second-order PT) 1.73177083 [E_ref - 7/24 g^2 = 1.73177083], iterate 5 1.73045994, iterate 20 1.73045985, iterate 80 1.73045985, FCI 1.73045985
g = 0.5: iterate 1 (second-order PT) 1.42708333 [E_ref - 7/24 g^2 = 1.42708333], iterate 5 1.41678269, iterate 20 1.41677428, iterate 80 1.41677428, FCI 1.41677428
g = 1.0: iterate 1 (second-order PT) 0.70833333 [E_ref - 7/24 g^2 = 0.70833333], iterate 5 0.63594518, iterate 20 0.63554847, iterate 80 0.63554847, FCI 0.63554847


The first iterate is exactly second-order perturbation theory, $E_{\rm ref}-\tfrac{7}{24}g^2$ (pause \#6 of week 37), and the sequence converges to the FCI energy. Each further iterate substitutes the right-hand side into itself once more — the order counting of many-body perturbation theory, which chapter 7 will make precise. Ten lines of code, and no matrix ever diagonalised, but the loop still runs over all $\binom{n}{N}$ amplitudes: the wall is untouched. The practical methods truncate the *equations*, not only the basis.

## 8. Hartree-Fock: optimise the reference instead of the expansion (chapter 6, sections 6.1-6.5)

The energy of one determinant is the functional $E[\Phi]=\sum_i\langle i|h_0|i\rangle+\tfrac12\sum_{ij}\langle ij|v|ij\rangle$ — $E_{\rm ref}$ of week 37. Expanding the orbitals in the fixed basis, $\psi_p=\sum_\alpha C_{p\alpha}\phi_\alpha$, and making $E$ stationary under the orthonormality constraint gives the Hartree-Fock equations $\sum_\beta h^{\rm HF}_{\alpha\beta}C_{i\beta}=\varepsilon_iC_{i\alpha}$ with $h^{\rm HF}_{\alpha\beta}=\langle\alpha|h_0|\beta\rangle+\sum_{\gamma\delta}\rho_{\gamma\delta}\langle\alpha\gamma|v|\beta\delta\rangle$ and the density matrix $\rho_{\gamma\delta}=\sum_iC_{i\gamma}C^*_{i\delta}$. The self-consistent field loop below runs on the random $n=8$, $N=4$ Hamiltonian of section 4 and checks: (i) $E^{\rm HF}\le E_{\rm ref}$ and $E^{\rm HF}\ge E_0^{\rm FCI}$; (ii) at convergence $f_{ai}=0$ (Brillouin); (iii) transforming $h$ and $v$ to the Hartree-Fock basis and rerunning FCI gives the *same* $E_0$ (the exact answer does not depend on the basis) but a Hamiltonian matrix whose $0p0h$-$1p1h$ block is empty; (iv) the correlation energy is then carried by the doubles alone in the energy equation.

In [14]:
def hartree_fock(h, vAS, N, nit=200, tol=1e-10, mix=1.0):
    n = h.shape[0]
    C = np.eye(n)                                  # C[:, i] = coefficients of orbital i (columns)
    eps_old = np.diag(h).copy()
    for it in range(nit):
        rho = C[:, :N] @ C[:, :N].T               # rho_{gamma delta} = sum_i C_{gamma i} C_{delta i}
        hHF = h + np.einsum('gd,agbd->ab', rho, vAS)
        eps, Cnew = np.linalg.eigh(hHF)
        C = Cnew
        drift = np.abs(eps - eps_old).mean(); eps_old = eps
        if drift < tol:
            break
    rho = C[:, :N] @ C[:, :N].T
    EHF = np.einsum('ab,ab', rho, h) + 0.5 * np.einsum('ac,bd,abcd', rho, rho, vAS)
    return EHF, eps, C, it + 1

EHF, epsHF, CHF, nits = hartree_fock(h8, v8, N4)
print(f"reference energy E_ref = {E_ref:.8f}")
print(f"Hartree-Fock energy    = {EHF:.8f}   after {nits} iterations")
print(f"FCI ground state       = {E8[0]:.8f}")
print(f"E_FCI <= E_HF <= E_ref :", E8[0] <= EHF + 1e-12 <= E_ref + 1e-12)
# Brillouin: transform h and v to the HF basis, build the Fock matrix there, check f_ai = 0
hT = CHF.T @ h8 @ CHF
vT = np.einsum('ap,bq,cr,ds,abcd->pqrs', CHF, CHF, CHF, CHF, v8)
fT = hT + sum(vT[:, i, :, i] for i in holes)
print(f"max |f_ai| in the HF basis     = {np.abs(fT[np.ix_(parts, holes)]).max():.1e}   (Brillouin's theorem)")
print(f"f diagonal in the HF basis? max off-diagonal |f_pq| = {np.abs(fT - np.diag(np.diag(fT))).max():.1e}; diagonal = HF orbital energies:", np.allclose(np.diag(fT), epsHF))
print(f"E_HF = sum_i eps_i - 1/2 sum_ij <ij|v|ij> = {epsHF[:N4].sum() - 0.5 * sum(vT[i, j, i, j] for i in holes for j in holes):.8f}")

# FCI in the HF basis: same energy, emptier matrix
HT, keepT = basis8.hamiltonian(hT, vT)
ET, CT = np.linalg.eigh(HT)
print(f"\nFCI in the HF basis: E_0 = {ET[0]:.8f}  (original basis {E8[0]:.8f})")
print("non-zero blocks in the HF basis:")
for k in range(5):
    print("  ", k, ["x" if np.abs(HT[np.ix_(lev8 == k, lev8 == l)]).max() > 1e-10 else "0" for l in range(5)])
vecT = CT[:, 0] / CT[0, 0]
E_singles = sum(fT[i, a] * vecT[basis8.index[det_of([a], [i])]] * sign_of([a], [i]) for a in parts for i in holes)
E_doubles = sum(vT[i, j, a, b] * vecT[basis8.index[det_of([a, b], [i, j])]] * sign_of([a, b], [i, j])
                for a, b in combinations(parts, 2) for i, j in combinations(holes, 2))
print(f"energy equation in the HF basis: Delta E = {ET[0] - EHF:.8f} = singles {E_singles:.1e} + doubles {E_doubles:.8f}")
truncT = {name: np.linalg.eigvalsh(basis8.hamiltonian(hT, vT, lv)[0])[0] for name, lv in {"CIS": {0, 1}, "CID": {0, 2}, "CISD": {0, 1, 2}}.items()}
trunc0 = {name: np.linalg.eigvalsh(basis8.hamiltonian(h8, v8, lv)[0])[0] for name, lv in {"CIS": {0, 1}, "CID": {0, 2}, "CISD": {0, 1, 2}}.items()}
print("truncated CI, original basis:", {k: round(x, 6) for k, x in trunc0.items()})
print("truncated CI, HF basis      :", {k: round(x, 6) for k, x in truncT.items()}, " FCI:", round(ET[0], 6))

reference energy E_ref = 6.49717364
Hartree-Fock energy    = 1.29801640   after 48 iterations
FCI ground state       = 0.74771322
E_FCI <= E_HF <= E_ref : True


max |f_ai| in the HF basis     = 8.8e-11   (Brillouin's theorem)
f diagonal in the HF basis? max off-diagonal |f_pq| = 8.8e-11; diagonal = HF orbital energies: True
E_HF = sum_i eps_i - 1/2 sum_ij <ij|v|ij> = 1.29801640



FCI in the HF basis: E_0 = 0.74771322  (original basis 0.74771322)
non-zero blocks in the HF basis:
   0 ['x', '0', 'x', '0', '0']
   1 ['0', 'x', 'x', 'x', '0']
   2 ['x', 'x', 'x', 'x', 'x']
   3 ['0', 'x', 'x', 'x', 'x']
   4 ['0', '0', 'x', 'x', 'x']
energy equation in the HF basis: Delta E = -0.55030318 = singles 5.3e-12 + doubles -0.55030318


truncated CI, original basis: {'CIS': np.float64(2.438662), 'CID': np.float64(5.442075), 'CISD': np.float64(1.157492)}
truncated CI, HF basis      : {'CIS': np.float64(1.298016), 'CID': np.float64(0.819738), 'CISD': np.float64(0.774941)}  FCI: 0.747713


CIS in the Hartree-Fock basis returns exactly $E^{\rm HF}$ — Brillouin's theorem says the reference is decoupled from the singles — and CID in the HF basis is lower than CID in the original basis: the truncations depend on the reference, the full CI does not. That is the bridge from this week to the next: the best single determinant first, the correlations on top of it afterwards.